# Neural Networks: Assignment week 4
Nea Lukumies

Assignment instructions: https://github.com/kopuj/neuralNetworks/blob/master/Assignments/Assignment_week4.md

The goal of this assignment is to understand how word embeddings and vectorization work. The embeddings used are from the GloVe project (glove 2024 wikigiga 300d). First the embeddings are downloaded locally and then checked that the embeddings are working as expected. 

In [62]:
import numpy as np

filepath = "./data/glove.2024.wikigiga.300d/wiki_giga_2024_300_MFT20_vectors_seed_2024_alpha_0.75_eta_0.05_combined.txt"
with open(filepath, "r", encoding="utf-8") as f:
    for i, line in enumerate(f):
        parts = line.strip().split()
        try:
            _ = list(map(float, parts[1:]))
        except:
            print("Error on line", i)
            print(line[:200])
            break


Error on line 4927
. . . 0.11100499999999999 0.032627 0.180207 0.069165 -0.160399 0.210655 0.164412 0.149789 0.056175 -0.173293 0.158837 0.026561 0.010095 0.272144 -0.160776 -0.089427 -0.201361 -0.052506999999999984 0.0


Embeddings are loaded into a dictionary and value errors are handled by continuing to the next line (issue seen on output above). 

In [63]:
DIM = 300
embeddings = {}

with open(filepath, "r", encoding="utf-8") as f:
    for line in f:
        parts = line.strip().split()

        # keep only correctly sized lines
        if len(parts) != DIM + 1:
            continue

        word = parts[0]

        try:
            vector = np.array(parts[1:], dtype=float)
            embeddings[word] = vector
        except ValueError:
            continue

print("Loaded words:", len(embeddings))
print("Dimension:", len(embeddings["king"]))

Loaded words: 1287614
Dimension: 300


The embeddins now include 300-dimensional vectors for 1,2 million words. To determine the nearest words to another, the cosine similarity is calculated between the vectors (embeddings) of the words. That means calculation the product of the vectors and dividing it by the product of the vector´s norms (lengths). The nearest words are then sorted by their cosine similarity to the target word (the closer the cosine similarity is to 1, the more similar the words are).

In [64]:
def cosine_similarity(vec1, vec2):
    dot_product = np.dot(vec1, vec2)
    norm_vec1 = np.linalg.norm(vec1)
    norm_vec2 = np.linalg.norm(vec2)

    if norm_vec1 == 0 or norm_vec2 == 0:
        return 0.0

    return dot_product / (norm_vec1 * norm_vec2)

The assignment was to calculate the vector obtained from the following expression:
vec("woman") - vec("man") + vec("king")
And then find the nearest words to this vector. I am for interests sake printing out the 10 nearest words to the resulting vector. Below is first the resulting vector in its numerical form and then the 10 nearest words to it.

In [65]:
embeddings["woman"] - embeddings["man"] + embeddings["king"]

array([-1.844590e-01, -4.232220e-01,  9.110600e-02, -1.051715e+00,
       -6.264410e-01, -5.166560e-01,  8.308580e-01, -4.537080e-01,
       -3.520690e-01,  4.618600e-01, -1.510790e-01,  9.727290e-01,
        1.828840e-01,  1.831470e-01, -7.136980e-01, -1.173400e-02,
       -2.392200e-02,  3.942640e-01,  4.744140e-01, -8.122960e-01,
       -4.023830e-01, -6.468840e-01,  2.385900e-01, -4.353440e-01,
       -6.623230e-01,  6.077870e-01,  7.881760e-01,  5.709700e-01,
       -2.145390e-01, -4.620400e-02, -7.207500e-02, -5.042770e-01,
        1.925510e-01,  2.510980e-01, -2.738610e-01,  6.751820e-01,
       -1.157422e+00,  5.025110e-01, -2.365470e-01, -7.089890e-01,
       -4.590300e-02,  9.452430e-01, -3.768960e-01,  1.336850e-01,
        2.689410e-01,  1.089383e+00, -3.891030e-01, -6.672400e-02,
        6.068200e-02,  7.573720e-01, -7.647470e-01,  3.740840e-01,
       -1.277760e-01,  1.612920e-01, -7.266600e-02,  5.355930e-01,
       -4.313010e-01,  8.183640e-01,  1.443450e-01, -3.067300e

In [66]:
nearest_words = sorted(embeddings.keys(), key=lambda w: cosine_similarity(embeddings["woman"] - embeddings["man"] + embeddings["king"], embeddings[w]), reverse=True)
print(nearest_words[:10])

['king', 'queen', 'princess', 'throne', 'monarch', 'mother', 'daughter', 'elizabeth', 'wife', 'kingdom']


The closest is in fact "king" and the second closest is "queen", meaning that the vector for "king" is more similar to the resulting vector than other word. Other words are more feminine making sense as the vector for "man" is subtracted. It seems that the vector as the expression is, if you add a vector for a word, that word or its plural form is likely to be the closest word to the resulting vector.

In the following I am testing out some other expressions to find interesting results.

In [67]:
def top_k_words(vec, embeddings, k=10, exclude=None):
    if exclude is None:
        exclude = set()

    scores = []

    for word, v in embeddings.items():
        if word in exclude:
            continue

        score = cosine_similarity(vec, v)
        scores.append((word, score))

    scores.sort(key=lambda x: x[1], reverse=True)
    return scores[:k]

cases = [
    ("king - man + woman", "king", "man", "woman"),
    ("queen - woman + man", "queen", "woman", "man"),
    ("spring - winter + summer", "spring", "winter", "summer"),
    ("easter - spring + holiday", "easter", "spring", "holiday"),
    ("woman - man + nature", "woman", "man", "nature"),
    ("forest - tree + river", "forest", "tree", "river"),
    ("city - town + village", "city", "town", "village"),
    ("car - road + air", "car", "road", "air"),
    ("music - sound + silence", "music", "sound", "silence"),
    ("computer - keyboard + mouse", "computer", "keyboard", "mouse"),
    ("python - snake + programming", "python", "snake", "programming"),
    ("doctor - hospital + school", "doctor", "hospital", "school")
]
for expr, a, b, c in cases:

    if a not in embeddings or b not in embeddings or c not in embeddings:
        print("\nExpression:", expr)
        print("Skipping (missing word)")
        continue

    vec = embeddings[a] - embeddings[b] + embeddings[c]

    exclude = {a, b, c}

    results = top_k_words(vec, embeddings, k=10, exclude=exclude)

    print("\n" + "="*60)
    print("Expression:", expr)
    print("="*60)

    for i, (word, score) in enumerate(results, 1):
        print(f"{i:2d}. {word:15s}  cosine similarity: {score:.4f}")


Expression: king - man + woman
 1. queen            cosine similarity: 0.7276
 2. princess         cosine similarity: 0.6061
 3. throne           cosine similarity: 0.5993
 4. monarch          cosine similarity: 0.5973
 5. mother           cosine similarity: 0.5888
 6. daughter         cosine similarity: 0.5831
 7. elizabeth        cosine similarity: 0.5721
 8. wife             cosine similarity: 0.5649
 9. kingdom          cosine similarity: 0.5593
10. her              cosine similarity: 0.5538

Expression: queen - woman + man
 1. king             cosine similarity: 0.7047
 2. ii               cosine similarity: 0.5747
 3. royal            cosine similarity: 0.5555
 4. prince           cosine similarity: 0.5520
 5. monarch          cosine similarity: 0.5514
 6. vi               cosine similarity: 0.5302
 7. majesty          cosine similarity: 0.5211
 8. elizabeth        cosine similarity: 0.5130
 9. victoria         cosine similarity: 0.5021
10. iii              cosine similarity: 0.

For example when turning the original expression woman- man + king into king - man + woman, the closest two are queen and princess and the word king is not now in the closest 10. Funnily enough, when using the exprssion queen - woman + man you also get i, vi and iii in the top 10 as closest words as the kings usually are associated with the roman numerals.

What interests me, is the fact that the cosine similarity for the top 10 closest words are all very close to each other, meaning that the differences are small. For example for the expression spring - winter + summer, the closest word is "fall" with 0.6398 cosine similarity and the next one is the word "beginning" with 0.6357 cosine similarity. With the expression easter - spring + holiday, the closest word is "christmas" with 0.6646 cosine similarity and the next one is "holidays" with 0.6544 cosine similarity. python - snake + programming resulted in words in programming and computer scienc with very small differences in cosine similarity.

Some cases are not that logical, for example the expression music - sound + silence gives the closest word "concert" and the next ones are "songs", "concerts" before reaching some more logical words such as "remembrance" and "mourning". All in all, the small differences in cosine similarity and the fact that the closest words are not always that logical, makes me think that the resulting vector is not very specific and can be close to many different words. Here maybe the ability to understand the expression and how to form it is very important to achieve good results and a more specific resulting vector. 